# Debugging Agents

**What you will build:** the habits for finding out *why* an agent misbehaved. Because agents are non-deterministic (0.0), you don't debug them by staring at code — you **inspect what actually happened**: the messages, the tool calls, the tokens. This is the code-course answer to n8n's "read the execution panel."

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/19_debugging_agents.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## The golden rule: read the transcript

When an agent does something wrong, the first question is always *what did the model actually see and do?* PydanticAI's `result.all_messages()` is the full record — the same message list you built by hand in Block 0. Let's debug a real failure with it.

You've already met bug #1 (a tool the model ignores — 1.2's vague description). Today's patient is the *nastier* cousin, bug #2: a tool the model calls **with silently wrong arguments**. The tool below expects `DD/MM/YYYY` — but nothing tells the model that:

In [ ]:
from datetime import date, datetime

broken = Agent(model, system_prompt="You are a scheduling assistant. Use tools when they help.")

@broken.tool_plain
def days_until(target_date: str) -> str:
    """How many days from today until the given date."""      # <- says nothing about the FORMAT
    target = datetime.strptime(target_date, "%d/%m/%Y").date()   # expects DD/MM/YYYY
    return f"{(target - date.today()).days} days"

result = await broken.run("How many days until March 5th, 2027?")
print("answer:", result.output)

Depending on the model you'll see one of two outcomes — and the *quiet* one is worse. If it passed ISO format (`2027-03-05`), `strptime` raised and the agent apologized: annoying but visible. If it passed US format (`03/05/2027`), the tool **parsed it as the 3rd of May** and returned a number that's wrong by two months — no error, confident answer, silent damage. Don't guess which happened — inspect. The helper from 1.2, now a permanent habit:

In [ ]:
def show_tool_calls(result):
    calls = [(p.tool_name, p.args) for m in result.all_messages()
             for p in m.parts if getattr(p, 'part_kind', '') == 'tool-call']
    return calls or "NO TOOLS CALLED — the model answered on its own"

print(show_tool_calls(result))

There it is, in the `args`: the model guessed a format, because the docstring never said which one. The transcript turns "the date math is flaky" into a one-line diagnosis. Two fixes, and do both: **say the format in the schema** (with an example — 1.2's rule), and **validate inside the tool** so a wrong guess becomes feedback instead of silence (1.2's `ModelRetry`):

In [ ]:
from pydantic_ai import ModelRetry

fixed = Agent(model, system_prompt="You are a scheduling assistant. Use tools when they help.")

@fixed.tool_plain
def days_until(target_date: str) -> str:
    """How many days from today until a date. target_date MUST be ISO format: YYYY-MM-DD, e.g. '2027-03-05'."""
    try:
        target = date.fromisoformat(target_date)
    except ValueError:
        raise ModelRetry(f"{target_date!r} is not ISO format. Send YYYY-MM-DD, e.g. '2027-03-05'.")
    return f"{(target - date.today()).days} days"

result = await fixed.run("How many days until March 5th, 2027?")
print("answer:", result.output)
print("tools called:", show_tool_calls(result))

Check the `args` now: unambiguous ISO, verifiable at a glance — and if the model ever slips, the `ModelRetry` turns the slip into a correction instead of a wrong answer. You found all of it by **evidence**, not intuition. That's the whole skill.

```{tip}
Nine out of ten agent bugs are one of: (1) a tool that wasn't called (bad description — 1.2), (2) a tool called with bad arguments (ambiguous schema — this one), or (3) a tool that returned something confusing. `all_messages()` shows you which, instantly.
```

## Watch the cost, too

A runaway agent shows up in the token count before you notice anything else. `result.usage` totals the tokens across the whole run (all loop steps).

In [ ]:
u = result.usage
print("tokens this run:", u.total_tokens)
print("(a sudden jump here usually means the agent is looping or re-reading a huge context)")

## When the run *dies*: `capture_run_messages`

Everything so far inspected runs that finished. But 1.5 showed you runs that don't — retries exhausted, `UnexpectedModelBehavior` raised, and… no `result` object to call `all_messages()` on. The transcript you need most is from the run you can't ask for it.

`capture_run_messages` solves exactly this: wrap the run, and the messages survive the crash:

In [ ]:
from pydantic_ai import capture_run_messages, ModelRetry
from pydantic_ai.exceptions import UnexpectedModelBehavior

doomed = Agent(model, retries=1, system_prompt="You name products.")

@doomed.output_validator
def never_happy(output: str) -> str:
    raise ModelRetry("Not good enough. Try again.")     # unsatisfiable, like in 1.5

with capture_run_messages() as messages:
    try:
        await doomed.run("Name a coffee brand.")
    except UnexpectedModelBehavior as e:
        print(f"run died: {e}\n\npost-mortem transcript:")
        for m in messages:
            for p in m.parts:
                print(f"  {p.part_kind:13s} {str(getattr(p, 'content', ''))[:70]}")

The post-mortem reads like a flight recorder: attempt, retry feedback, attempt, boom. In this case the transcript indicts the *validator* (the feedback "Not good enough" gives the model nothing to fix — 1.5's lesson), which no amount of staring at the exception message would have told you. In production code, that `with capture_run_messages():` block plus a log statement in the `except` is cheap insurance around any agent call.

## Not all failures are the same: a retry taxonomy

You've now met two kinds of retry — worth naming the whole family, because each failure needs a *different* fix. Reaching for `ModelRetry` when the real problem is a flaky network just hides a rate-limit inside a validation loop.

| Failure | What it looks like | Where it's handled | Right recovery |
|---|---|---|---|
| **Validation** | output doesn't fit the schema | `output_validator` / `output_type` (1.3, 1.5) | `ModelRetry` — feed the error back, let the model fix it |
| **Bad tool argument** | tool called with a nonsensical value | inside the tool (1.2) | `ModelRetry` — say what a valid argument looks like |
| **Transport** | connection reset, 500/503, timeout | the HTTP client / provider layer | retry with backoff — *not* `ModelRetry`; the model did nothing wrong |
| **Rate limit** | HTTP 429, "too many requests" | the provider layer | back off and slow down; `UsageLimits` (1.5b) helps you stay under quota |
| **Model / provider outage** | one model or vendor is down or refuses | agent construction | **fall back** to another model or provider |

The top two you already handle with `ModelRetry`. The bottom three are *infrastructure* failures — the model isn't at fault, so a self-correction prompt is the wrong tool. Transport and rate-limit retries belong to the client/provider layer (backoff, not re-prompting). The outage case has a first-class answer in PydanticAI: `FallbackModel`.

In [ ]:
from pydantic_ai.models.fallback import FallbackModel

# Try the primary model; if it errors (outage, 5xx, refusal), transparently try the next.
backup = OpenAIChatModel(
    "openai/gpt-4o-mini",                         # any second model/provider you trust
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
robust = Agent(FallbackModel(model, backup),      # model = the primary from the setup cell
               system_prompt="You are a concise assistant.")

r = await robust.run("In one sentence, what is a fallback model for?")
print(r.output)
# If EVERY model fails, PydanticAI raises FallbackExceptionGroup with each underlying error
# (catch it with Python 3.11+ `except*`).

`FallbackModel` covers the last row — vendor outages and hard refusals — by moving to the next model without touching your agent code. The middle rows (transport, rate limit) are the provider client's job: retry with backoff, respect `Retry-After`. The point of the table is diagnostic discipline: **name the failure before you pick the recovery.** A 429 is not a validation error, and no amount of re-prompting fixes a network reset.

## Tracing: see every step, automatically

Printing `all_messages()` works, but for real systems you want a trace UI that records *every* run without you asking. That's **Logfire**, which you already met in 1.6 (`logfire.configure()` + `logfire.instrument_pydantic_ai()` — two lines, free account). Nothing new to learn here: the trace UI shows the same parts vocabulary you've been reading all chapter, permanently, for every run.

```{note}
**LangGraph/LangChain** have the equivalent in **LangSmith**: set two environment variables and traces appear automatically — no code change.

```python
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "..."   # from smith.langchain.com
```

Same idea as Logfire: stop guessing, look at the trace.
```

## A debugging checklist

| Symptom | First thing to check | Likely fix |
|---------|----------------------|-----------|
| Wrong / made-up answer | `all_messages()` — was a tool called? | Sharpen the tool description (1.2) |
| Tool called with bad args | The `tool-call` part's `args` | Add an example to the docstring |
| Runs forever / expensive | `usage.total_tokens` per run | Add a turn/recursion cap; check for loops |
| Ignores an instruction | The system prompt in the transcript | Tighten/de-conflict the rules (appendix) |
| Fails only sometimes | Run it 5x; it's non-determinism | Lower `temperature`; add an eval (1.6) |

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Diagnose a bad-args bug of your own.** Give a tool an argument named `x` with no description, ask a question that needs it, and read the `tool-call` part's `args` to see what the model guessed. **Done when:** you've predicted the fix before writing it.
2. **Catch a loop.** Build an agent whose tool always answers "not done yet, try again" and watch `usage.total_tokens` climb — then stop it properly with `UsageLimits(request_limit=4)` (1.5b) and catch the exception. **Done when:** the run ends with `UsageLimitExceeded` instead of a huge bill.
3. **Post-mortem practice.** Take the exhausted-retries agent from 1.5 and, using `capture_run_messages`, write down from the transcript alone: how many attempts, what feedback each got, and what you'd change. **Done when:** your diagnosis names the validator's feedback, not the model, as the bug.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**2 — Catch a loop:**

```python
from pydantic_ai.usage import UsageLimits
from pydantic_ai.exceptions import UsageLimitExceeded

looper = Agent(model, system_prompt="Use tools to finish the task.")

@looper.tool_plain
def try_again(task: str) -> str:
    """Attempt a task. May need several attempts."""
    return "Not done yet — call try_again once more."

try:
    await looper.run("Finish the task with try_again.",
                     usage_limits=UsageLimits(request_limit=4))
except UsageLimitExceeded as e:
    print("caught:", e)
```

The tool *invites* an infinite loop; the limit converts it into a clean, catchable failure. Every production run should carry one — it's the typed version of 0.3's `max_turns`.

**3 —** The transcript shows: attempt 1 (`text`), `retry-prompt` ("Not good enough. Try again."), attempt 2, exception. Diagnosis: the model produced two perfectly reasonable coffee brands; the *validator's feedback* named no criterion, so no third attempt could have done better. Fix the feedback (name the rule), not the model — the same conclusion the flight recorder gave us above, reached from evidence.
::::

**Block 1 complete.** You can build, test (1.5b), evaluate (1.6), and now debug clean typed agents — the toolkit for most real applications.

**What's next — Block 2:** when a task needs explicit **state, branching, cycles, or multiple agents**, you graduate to **LangGraph**. First stop, **2.0**: the same agent as a graph — and you finally get a picture to look at.